# Notebook 03 — Construction du panel final (affichage des résultats)

> ⚠️ **Ce notebook ne calcule plus rien.** Depuis la réorganisation du projet, tout le
> calcul est fait par le script `scripts/etape03_construction_panel.py`, à lancer **à la main**, depuis la racine du
> projet :
>
> ```bash
> python scripts/etape03_construction_panel.py
> ```
>
> Ce notebook se contente de **lire et afficher** ce que ce script a produit : les gros
> fichiers de sortie (chemins dans `config.py`) et le rapport d'exécution `03_panel`
> (`outputs/rapports/`, voir `rapports.py`), qui contient tous les compteurs et petits
> tableaux de diagnostic autrefois imprimés au fil des cellules.
>
> Il est donc **léger et ré-exécutable à volonté** (`Run All` en quelques secondes), sans
> jamais relancer un calcul. Si tu changes un paramètre dans `config.py`, relance d'abord
> le script, puis ce notebook.


**Ce que le script a fait**, en 2 parties **séquentielles** (la partie B part du panel
construit par la partie A, sans repasser par le disque) :

- **Partie A — Fusion** : assembler les 3 fichiers nettoyés à l'étape 02, avec
  l'alignement temporel correct pour chaque source, et calculer la cible
  `excess_return = RET - Rfree` → `data/processed/panel_final.parquet`
- **Partie B — Préparation pour la modélisation** : filtres taille/liquidité, imputation,
  winsorizing, rank transform → `data/processed/panel_pret_modelisation.parquet`

💡 Pour ne rejouer que la partie B (test d'un nouveau seuil de filtrage), sans refaire la
fusion : `python scripts/etape03_construction_panel.py --partie-b-seulement`.

## 0. Import et chargement du rapport d'exécution

In [1]:
import sys
sys.path.append("..")

import pandas as pd

import config
import rapports

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

rap = rapports.charger('03_panel')
print(rap.resume())

Rapport '03_panel' produit le 2026-08-16T14:29:51


## Partie A — Fusion des 3 bases nettoyées

ℹ️ **L'alignement temporel est le point clé de cette partie.** `datashare.parquet` (Dacheng
Xiu) est **déjà pré-décalé** dans le fichier source : pour une ligne `DATE=19570329`, le
`RET` de 195703 est directement la bonne variable à expliquer. **Aucun décalage
supplémentaire n'est donc nécessaire** entre caractéristiques et rendements.

En revanche, les **prédicteurs macro** (Welch-Goyal) ne sont **pas** pré-décalés : le
script les décale lui-même d'un mois (le prédicteur du mois *m* sert à prédire *m+1*).
`Rfree`, lui, n'est **pas** décalé : il sert à calculer le rendement excédentaire du
**même** mois que `RET`.

### A.1 / A.3 Chevauchement avant fusion

C'est l'étape qu'on ne doit **jamais sauter** : si le chevauchement est faible ou nul, la
fusion produit un panel vide ou minuscule.

In [2]:
print("Caracteristiques :", tuple(rap.valeur('A_shape_chars')))
print("Rendements       :", tuple(rap.valeur('A_shape_returns')))
print("Macro            :", tuple(rap.valeur('A_shape_macro')))
print()
print(f"Entreprises dans characteristics_clean : {rap.valeur('A_n_entreprises_chars')}")
print(f"Entreprises dans returns_clean         : {rap.valeur('A_n_entreprises_returns')}")
print(f"Entreprises communes                   : {rap.valeur('A_n_entreprises_communes')}")
print(f"-> {rap.valeur('A_pct_entreprises_communes'):.1f}% des entreprises de chars ont au moins un rendement")
print()
for nom, cle in [("chars  ", 'A_periodes_chars'), ("returns", 'A_periodes_returns'), ("macro  ", 'A_periodes_macro')]:
    n, mini, maxi = rap.valeur(cle)
    print(f"Periodes {nom} : {n} mois ({mini} a {maxi})")

Caracteristiques : (3344804, 34)
Rendements       : (4669780, 3)
Macro            : (552, 10)

Entreprises dans characteristics_clean : 29731
Entreprises dans returns_clean         : 38297
Entreprises communes                   : 29709
-> 99.9% des entreprises de chars ont au moins un rendement

Periodes chars   : 504 mois (198001 a 202112)
Periodes returns : 828 mois (195601 a 202412)
Periodes macro   : 552 mois (198001 a 202512)


### A.4 / A.5 Les deux fusions

Fusion 1 (**inner** sur `permno` + `annee_mois`) : on ne garde que les lignes où on a
**à la fois** les caractéristiques et le rendement du même mois — une ligne sans l'un des
deux n'est pas utilisable pour entraîner un modèle supervisé.

Fusion 2 (**left**, en deux temps) : les 8 prédicteurs macro décalés d'un mois, puis
`Rfree` non décalé. Les lignes sans correspondance macro (inévitables pour le tout premier
mois du panel) sont retirées.

In [3]:
print(f"Lignes characteristics avant fusion : {rap.valeur('A_lignes_chars_avant_fusion')}")
print(f"Lignes returns avant fusion         : {rap.valeur('A_lignes_returns_avant_fusion')}")
print(f"Lignes apres fusion 1 (intersection): {rap.valeur('A_lignes_apres_fusion1')}")
print()
print(f"Lignes apres fusion predicteurs macro : {rap.valeur('A_lignes_apres_fusion_predicteurs_macro')}")
print(f"Lignes apres fusion Rfree             : {rap.valeur('A_lignes_apres_fusion_rfree')}")
print()
print(f"Lignes sans donnees macro correspondantes : {rap.valeur('A_lignes_sans_macro')} "
      f"({rap.valeur('A_pct_lignes_sans_macro'):.2f}%) -> retirees")
print(f"Lignes restantes : {rap.valeur('A_lignes_apres_suppression_sans_macro')}")

Lignes characteristics avant fusion : 3344804
Lignes returns avant fusion         : 4669780
Lignes apres fusion 1 (intersection): 3326704

Lignes apres fusion predicteurs macro : 3326704
Lignes apres fusion Rfree             : 3326704

Lignes sans donnees macro correspondantes : 4764 (0.14%) -> retirees
Lignes restantes : 3321940


### La variable cible : le rendement excédentaire

Suivant Gu, Kelly & Xiu (2020), la variable à prédire n'est pas le rendement brut `RET`
mais le **rendement excédentaire** par rapport au taux sans risque :
`excess_return = RET - Rfree`.

In [4]:
display(rap.table('A_apercu_cible'))
rap.table('A_describe_cible')

,permno,annee_mois,RET,Rfree,excess_return
4764,10006,198002,-0.058795,0.0089,-0.067695
4765,10057,198002,-0.183582,0.0089,-0.192482
4766,10058,198002,0.000000,0.0089,-0.008900
4767,10065,198002,-0.029126,0.0089,-0.038026
4768,10103,198002,-0.230769,0.0089,-0.239669


,excess_return
count,3.321940e+06
mean,7.457279e-03
std,1.822209e-01
min,-9.952000e-01
25%,-6.510000e-02
50%,-1.785000e-03
75%,6.276700e-02
max,2.399660e+01


### A.6 / A.7 Vérifications finales et panel fusionné

In [5]:
print(f"Doublons (permno, annee_mois) restants : {rap.valeur('A_doublons_restants')}")
print()
print("Dimensions finales du panel :", tuple(rap.valeur('A_shape_panel_final')))
print("Nombre d'entreprises uniques :", rap.valeur('A_n_entreprises_panel'))
print("Periode couverte : de", rap.valeur('A_periode_panel')[0], "a", rap.valeur('A_periode_panel')[1])
print(f"Memoire du panel (apres passage de permno/annee_mois en 'category') : "
      f"{rap.valeur('A_memoire_panel_mo'):.1f} Mo")
print()
manquants = rap.table('A_missing_panel')
print("Valeurs manquantes restantes :")
display(manquants if len(manquants) else "Aucune valeur manquante.")
print("Fichier ecrit par le script :", config.FICHIER_PANEL_FINAL)

Doublons (permno, annee_mois) restants : 0

Dimensions finales du panel : (3321940, 48)
Nombre d'entreprises uniques : 29657
Periode couverte : de 198002 a 202112
Memoire du panel (apres passage de permno/annee_mois en 'category') : 1243.7 Mo

Valeurs manquantes restantes :


,nb_manquant
mom1m,24733
mom6m,122705
mom12m,265103
chmom,265103
indmom,9
maxret,202
mvel1,1097
dolvol,139964
turn,142005
std_turn,98453


Fichier ecrit par le script : C:\Users\aless\OneDrive\Documents\Pantheon Sorbonne\cour\Memoire\Memoire\data\processed\panel_final.parquet


## Partie B — Préparation pour la modélisation

### B.2 Filtrage de l'univers investissable (taille puis liquidité)

Deux filtres successifs, **mois par mois**, avant toute imputation ou winsorisation :
d'abord la **taille** (`mvel1`, on exclut sous le percentile
`config.SEUIL_PERCENTILE_TAILLE`), puis la **liquidité** (`ill`, critère d'Amihud, on
exclut au-dessus de `config.SEUIL_PERCENTILE_LIQUIDITE`).

**Pourquoi mois par mois** (jamais sur une moyenne de vie entière d'une entreprise) :
(1) une moyenne calculée sur toute la période inclurait de l'information du futur pour une
décision qui affecte des lignes passées — une fuite de données — et (2) ça exclurait
*toute* une entreprise, y compris ses mois où elle était grande et fiable.

ℹ️ `mvel1` et `ill` ne sont **jamais imputés** : on ne filtre pas sur une valeur inconnue,
leurs lignes manquantes sont retirées juste avant le filtre correspondant.

In [6]:
print(f"Seuil taille (config.py)    : {rap.valeur('B_seuil_percentile_taille')}")
print(f"Seuil liquidite (config.py) : {rap.valeur('B_seuil_percentile_liquidite')}")
print()
print("--- Filtre TAILLE (mvel1) ---")
print(f"  mvel1 manquants retires : {rap.valeur('B_lignes_avant_dropna_mvel1')} -> "
      f"{rap.valeur('B_lignes_apres_dropna_mvel1')} lignes "
      f"({rap.valeur('B_entreprises_avant_dropna_mvel1')} -> {rap.valeur('B_entreprises_apres_dropna_mvel1')} entreprises)")
avant, apres = rap.valeur('B_lignes_avant_filtre_taille'), rap.valeur('B_lignes_apres_filtre_taille')
print(f"  Filtre de taille        : {avant} -> {apres} lignes ({(avant-apres)/avant*100:.2f}% retirees)")
print(f"  Entreprises             : {rap.valeur('B_entreprises_avant_filtre_taille')} -> "
      f"{rap.valeur('B_entreprises_apres_filtre_taille')}")
print()
print("--- Filtre LIQUIDITE (ill) ---")
print(f"  ill manquants retires   : {rap.valeur('B_lignes_avant_dropna_ill')} -> "
      f"{rap.valeur('B_lignes_apres_dropna_ill')} lignes")
avant, apres = rap.valeur('B_lignes_avant_filtre_liquidite'), rap.valeur('B_lignes_apres_filtre_liquidite')
print(f"  Filtre de liquidite     : {avant} -> {apres} lignes ({(avant-apres)/avant*100:.2f}% retirees)")
print(f"  Entreprises             : {rap.valeur('B_entreprises_avant_filtre_liquidite')} -> "
      f"{rap.valeur('B_entreprises_apres_filtre_liquidite')}")
print()
print(f"Effet CUMULE des deux filtres : {rap.valeur('B_pct_retire_total_filtres'):.2f}% des lignes retirees.")

Seuil taille (config.py)    : 0.1
Seuil liquidite (config.py) : 0.9

--- Filtre TAILLE (mvel1) ---
  mvel1 manquants retires : 3321940 -> 3320843 lignes (29657 -> 29652 entreprises)
  Filtre de taille        : 3320843 -> 2988598 lignes (10.00% retirees)
  Entreprises             : 29652 -> 28664

--- Filtre LIQUIDITE (ill) ---
  ill manquants retires   : 2988598 -> 2906647 lignes
  Filtre de liquidite     : 2906647 -> 2615807 lignes (10.01% retirees)
  Entreprises             : 28087 -> 27556

Effet CUMULE des deux filtres : 21.26% des lignes retirees.


### B.3 Imputation des valeurs manquantes restantes

Les caractéristiques autres que `mvel1` et `ill` peuvent encore contenir des valeurs
manquantes à ce stade (volontairement non imputées à l'étape 02). Elles le sont maintenant,
**après les filtres** : la médiane utilisée pour chaque mois doit venir de la population
qu'on garde réellement, pas d'une population plus large incluant des micro-caps et des
titres illiquides qu'on vient d'exclure.

Médiane du mois → médiane globale de la colonne en filet de sécurité → 0 en tout dernier
recours (colonne entièrement vide).

In [7]:
print("% de valeurs manquantes par caracteristique AVANT imputation :")
display(rap.table('B_missing_avant_imputation').head(20))

print(f"Imputation effectuee sur {rap.valeur('B_n_caracteristiques_imputees')} caracteristiques (hors mvel1).")
vides = rap.valeur('B_colonnes_entierement_vides')
if vides:
    print("ATTENTION - colonnes entierement vides sur la population post-filtres :", vides)
print(f"Valeurs manquantes restantes dans les caracteristiques : {rap.valeur('B_missing_restant_apres_imputation')}")

% de valeurs manquantes par caracteristique AVANT imputation :


,pct_manquant
acc,31.121715
invest,30.331060
chtx,29.822078
operprof,28.866847
roeq,27.262103
roaq,27.257821
gma,26.348504
chcsho,26.269866
cfp,26.246508
egr,26.237371


Imputation effectuee sur 29 caracteristiques (hors mvel1).
Valeurs manquantes restantes dans les caracteristiques : 0


### B.4 / B.5 Winsorizing puis transformation en rang

**Winsorizing** : chaque caractéristique est cappée à ses 1er et 99e centiles, calculés
séparément **pour chaque mois** — appliqué maintenant que les micro-caps et les titres
illiquides sont déjà filtrés (on filtre d'abord *qui* on garde, on nettoie ensuite *les
valeurs* de ce qu'on garde).

**Rank transform** : pour chaque caractéristique et chaque mois, la valeur brute est
remplacée par son **rang parmi les entreprises du même mois**, ramené dans `[-1, +1]`.

⚠️ Cette transformation n'utilise **que** les entreprises du même mois — aucune information
d'un autre mois (passé ou futur), donc **aucune fuite de données** entre train/validation/test.
C'est pour ça qu'elle peut être appliquée ici, avant tout découpage temporel.

Diagnostic attendu : un écart-type proche de **0.577** partout (l'écart-type d'une loi
uniforme sur `[-1, 1]`), et des bornes strictement dans `[-1, 1]`.

In [8]:
print("Ecart-type par caracteristique APRES rank transform (doit etre proche de 0.577) :")
display(rap.table('B_ecart_type_apres_rank'))

Ecart-type par caracteristique APRES rank transform (doit etre proche de 0.577) :


,ecart_type
chtx,0.565325
acc,0.567231
invest,0.568492
roeq,0.569453
roaq,0.569462
operprof,0.569592
cfp,0.571315
chcsho,0.571344
gma,0.571352
egr,0.571455


In [9]:
print("Min/Max apres transformation (doit etre entre -1 et 1) :")
rap.table('B_bornes_apres_rank')

Min/Max apres transformation (doit etre entre -1 et 1) :


,min,max
mom1m,-0.990338,0.990338
mom6m,-0.990338,0.990338
mom12m,-0.990338,0.990338
chmom,-0.990338,0.990338
indmom,-0.990220,0.990161
maxret,-0.990338,0.990338
mvel1,-0.990338,0.990338
dolvol,-0.990338,0.990338
turn,-0.990338,0.990338
std_turn,-0.990338,0.990338


### Variables macro : pas de standardisation ici

Les variables macro ont **une seule valeur par mois** et se standardisent classiquement en
z-score — mais cette moyenne et cet écart-type doivent être calculés **sur le train**, et
le train n'est pas unique : il change à chaque fenêtre. Le script sauvegarde donc les
variables macro **brutes** ; c'est `fenetres.preparer_fenetre` qui recalcule la
standardisation à la volée pour chaque fenêtre (étapes 04, 05, 06).

### B.7 Résultat final

In [10]:
print("Dimensions finales :", tuple(rap.valeur('B_shape_finale')))
print("Periode couverte   :", rap.valeur('B_periode')[0], "a", rap.valeur('B_periode')[1])
print("Nombre de mois distincts :", rap.valeur('B_n_mois_distincts'))
print()
manquants = rap.table('B_missing_final')
print("Valeurs manquantes restantes (doit etre 0) :")
display(manquants if len(manquants) else "Aucune valeur manquante.")
print("Fichier ecrit par le script :", config.FICHIER_PANEL_MODELISATION)
rap.table('B_apercu_final')

Dimensions finales : (2615807, 49)
Periode couverte   : 198002 a 202112
Nombre de mois distincts : 503

Valeurs manquantes restantes (doit etre 0) :


'Aucune valeur manquante.'

Fichier ecrit par le script : C:\Users\aless\OneDrive\Documents\Pantheon Sorbonne\cour\Memoire\Memoire\data\processed\panel_pret_modelisation.parquet


,permno,annee_mois,mvel1_brut,mom1m,mom6m,mom12m,chmom,indmom,macro_dp,macro_ep,excess_return
0,10006,198002,367648.5,0.706280,-0.241546,0.051208,-0.032850,-0.342995,-2.997135,-2.029331,-0.067695
1,10057,198002,142408.5,0.840580,0.690821,0.337198,0.893720,0.758937,-2.997135,-2.029331,-0.192482
3,10065,198002,201159.0,0.286957,0.057971,0.025121,0.107246,0.472464,-2.997135,-2.029331,-0.038026
5,10137,198002,597435.0,-0.685507,-0.589372,-0.472464,-0.621256,-0.484058,-2.997135,-2.029331,-0.075567
6,10145,198002,1627652.0,0.529469,0.754589,0.704348,0.722705,0.818357,-2.997135,-2.029331,-0.022174


## Résumé global et prochaines étapes

- **Partie A** : panel fusionné (caractéristiques + rendements + macro décalée + cible
  `excess_return`) → `data/processed/panel_final.parquet`
- **Partie B** : univers filtré (taille puis liquidité, mois par mois), valeurs manquantes
  imputées, caractéristiques winsorisées et rang-transformées →
  `data/processed/panel_pret_modelisation.parquet`, **sans découpage temporel ni
  standardisation macro** (les deux dépendent de la fenêtre, calculés à la volée par
  `fenetres.py`).

**Étape suivante :** `python scripts/etape04_modele_lineaire.py` (puis 05 et 06).